In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
df = pd.read_parquet('PERTA.parquet')

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

df["block_number"] = df["block_number_y"]

df = df.sort_values(["timestamp", "block_number", "transaction_index", "log_index"]).reset_index(drop=True)

In [ ]:
price_raw = []
for x in df["sqrtPriceX96"]:
    value = (float(x) / (2 ** 96)) ** 2
    price_raw.append(value)

df["price_raw"] = price_raw

price_token1_per_token0 = []
for x in df["price_raw"]:
    value = x * (10 ** (6 - 18))
    price_token1_per_token0.append(value)

df["price_token1_per_token0"] = price_token1_per_token0

price_usdc_per_weth = []
for x in df["price_token1_per_token0"]:
    if x == 0:
        price_usdc_per_weth.append(np.nan)
    else:
        price_usdc_per_weth.append(1.0 / x)

df["price_usdc_per_weth"] = price_usdc_per_weth

In [ ]:
gas_cost_eth = []
for i in range(len(df)):
    gas_used = float(df.loc[i, "gas_used"])
    gas_price = float(df.loc[i, "effective_gas_price"])
    value = gas_used * gas_price / 1e18
    gas_cost_eth.append(value)

df["gas_cost_eth"] = gas_cost_eth
df["hour"] = df["timestamp"].dt.floor("h")

In [ ]:
os.makedirs("figures", exist_ok=True)

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10

In [ ]:
hours = sorted(df["hour"].dropna().unique())

hour_list = []
swap_count_list = []

for h in hours:
    part = df[df["hour"] == h]
    hour_list.append(h)
    swap_count_list.append(len(part))

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(hour_list, swap_count_list)
ax.set_title("Swap activity over time")
ax.set_xlabel("Time")
ax.set_ylabel("Number of swap events per hour")

fig.savefig("figures/swap_activity_per_hour.png", bbox_inches="tight")

In [ ]:
hour_list = []
price_list = []

for h in hours:
    part = df[df["hour"] == h]
    value = part["price_usdc_per_weth"].median()
    hour_list.append(h)
    price_list.append(value)

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(hour_list, price_list)
ax.set_title("Pool price dynamics")
ax.set_xlabel("Time")
ax.set_ylabel("Price (USDC per WETH)")

fig.savefig("figures/pool_price_dynamics.png", bbox_inches="tight")


In [ ]:
hour_list = []
gas_list = []

for h in hours:
    part = df[df["hour"] == h]
    value = part["gas_cost_eth"].median()
    hour_list.append(h)
    gas_list.append(value)

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(hour_list, gas_list)
ax.set_title("Median gas cost over time")
ax.set_xlabel("Time")
ax.set_ylabel("Median gas cost (ETH)")

fig.savefig("figures/median_gas_cost_per_hour.png", bbox_inches="tight")


In [ ]:
hour_rows = []

for h in hours:
    part = df[df["hour"] == h]
    if len(part) == 0:
        continue

    row = {}
    row["hour"] = h
    row["open"] = part["price_usdc_per_weth"].iloc[0]
    row["close"] = part["price_usdc_per_weth"].iloc[-1]
    row["high"] = part["price_usdc_per_weth"].max()
    row["low"] = part["price_usdc_per_weth"].min()
    row["median_gas_eth"] = part["gas_cost_eth"].median()
    hour_rows.append(row)

hourly = pd.DataFrame(hour_rows)

rets = [0.0]
for i in range(1, len(hourly)):
    prev_price = hourly.loc[i - 1, "close"]
    cur_price = hourly.loc[i, "close"]
    if prev_price == 0 or pd.isna(prev_price) or pd.isna(cur_price):
        rets.append(0.0)
    else:
        rets.append((cur_price / prev_price) - 1.0)

hourly["ret"] = rets

In [ ]:
def run_ma_strategy(hourly, df):
    x = hourly.copy()

    ma_short = []
    ma_long = []

    for i in range(len(x)):
        if i < 23:
            ma_short.append(np.nan)
        else:
            value = x.loc[i-23:i, "close"].mean()
            ma_short.append(value)

    for i in range(len(x)):
        if i < 71:
            ma_long.append(np.nan)
        else:
            value = x.loc[i-71:i, "close"].mean()
            ma_long.append(value)

    x["ma_short"] = ma_short
    x["ma_long"] = ma_long

    signal = []
    for i in range(len(x)):
        s = x.loc[i, "ma_short"]
        l = x.loc[i, "ma_long"]
        if pd.isna(s) or pd.isna(l):
            signal.append(0)
        elif s > l:
            signal.append(1)
        elif s < l:
            signal.append(-1)
        else:
            signal.append(0)

    x["signal_ma"] = signal

    position = [0]
    for i in range(1, len(x)):
        position.append(x.loc[i - 1, "signal_ma"])
    x["position_ma_simple"] = position

    position_change = [0.0]
    trade_flag = [0]
    gross_ret = [0.0]
    cost = [0.0]
    strategy_ret = [0.0]
    equity = [1.0]

    for i in range(1, len(x)):
        change = abs(x.loc[i, "position_ma_simple"] - x.loc[i - 1, "position_ma_simple"])
        position_change.append(change)

        if change > 0:
            trade_flag.append(1)
        else:
            trade_flag.append(0)

        gr = x.loc[i, "position_ma_simple"] * x.loc[i, "ret"]
        gross_ret.append(gr)

        c = change * 0.0005
        cost.append(c)

        sr = gr - c
        strategy_ret.append(sr)

        eq = equity[-1] * (1.0 + sr)
        equity.append(eq)

    x["position_change_ma_simple"] = position_change
    x["trade_flag_ma_simple"] = trade_flag
    x["gross_ret_ma_simple"] = gross_ret
    x["cost_ma_simple"] = cost
    x["strategy_ret_ma_simple"] = strategy_ret
    x["equity_ma_simple"] = equity

    exec_rows = []
    seen = set()

    for i in range(len(df)):
        h = df.loc[i, "hour"]
        if h in seen:
            continue
        seen.add(h)
        row = {}
        row["exec_hour"] = h
        row["exec_price"] = df.loc[i, "price_usdc_per_weth"]
        row["exec_gas_cost"] = df.loc[i, "gas_cost_eth"]
        exec_rows.append(row)

    exec_events = pd.DataFrame(exec_rows)

    ma_real = x[["hour", "signal_ma"]].copy()
    real_position = [0]
    for i in range(1, len(ma_real)):
        real_position.append(ma_real.loc[i - 1, "signal_ma"])
    ma_real["position"] = real_position

    ma_real = ma_real.merge(exec_events, left_on="hour", right_on="exec_hour", how="left")

    exec_ret = [0.0]
    for i in range(1, len(ma_real)):
        prev_price = ma_real.loc[i - 1, "exec_price"]
        cur_price = ma_real.loc[i, "exec_price"]
        if prev_price == 0 or pd.isna(prev_price) or pd.isna(cur_price):
            exec_ret.append(0.0)
        else:
            exec_ret.append((cur_price / prev_price) - 1.0)
    ma_real["exec_ret"] = exec_ret

    position_change = [0.0]
    trade_flag = [0]
    gross_ret = [0.0]
    gas_cost_return = [0.0]
    slippage_cost_return = [0.0]
    cost = [0.0]
    strategy_ret = [0.0]
    equity = [1.0]

    for i in range(1, len(ma_real)):
        change = abs(ma_real.loc[i, "position"] - ma_real.loc[i - 1, "position"])
        position_change.append(change)

        if change > 0:
            trade_flag.append(1)
        else:
            trade_flag.append(0)

        gr = ma_real.loc[i, "position"] * ma_real.loc[i, "exec_ret"]
        gross_ret.append(gr)

        if change > 0 and not pd.isna(ma_real.loc[i, "exec_gas_cost"]):
            gas_val = ma_real.loc[i, "exec_gas_cost"] / 1.0
        else:
            gas_val = 0.0
        gas_cost_return.append(gas_val)

        slip = change * (1.0 / 10000.0)
        slippage_cost_return.append(slip)

        c = gas_val + slip
        cost.append(c)

        sr = gr - c
        strategy_ret.append(sr)

        eq = equity[-1] * (1.0 + sr)
        equity.append(eq)

    ma_real["position_change"] = position_change
    ma_real["trade_flag"] = trade_flag
    ma_real["gross_ret"] = gross_ret
    ma_real["gas_cost_return"] = gas_cost_return
    ma_real["slippage_cost_return"] = slippage_cost_return
    ma_real["cost"] = cost
    ma_real["strategy_ret"] = strategy_ret
    ma_real["equity"] = equity

    return x, ma_real

In [ ]:
def run_mr_strategy(hourly, df):
    x = hourly.copy()

    zscore = []
    for i in range(len(x)):
        if i < 47:
            zscore.append(np.nan)
        else:
            part = x.loc[i-47:i, "close"]
            mean_val = part.mean()
            std_val = part.std()
            if std_val == 0 or pd.isna(std_val):
                zscore.append(np.nan)
            else:
                z = (x.loc[i, "close"] - mean_val) / std_val
                zscore.append(z)

    x["zscore"] = zscore

    signal = []
    for i in range(len(x)):
        z = x.loc[i, "zscore"]
        if pd.isna(z):
            signal.append(0)
        elif z > 1.5:
            signal.append(-1)
        elif z < -1.5:
            signal.append(1)
        else:
            signal.append(0)

    x["signal_mr"] = signal

    position = [0]
    for i in range(1, len(x)):
        position.append(x.loc[i - 1, "signal_mr"])
    x["position_mr_simple"] = position

    position_change = [0.0]
    trade_flag = [0]
    gross_ret = [0.0]
    cost = [0.0]
    strategy_ret = [0.0]
    equity = [1.0]

    for i in range(1, len(x)):
        change = abs(x.loc[i, "position_mr_simple"] - x.loc[i - 1, "position_mr_simple"])
        position_change.append(change)

        if change > 0:
            trade_flag.append(1)
        else:
            trade_flag.append(0)

        gr = x.loc[i, "position_mr_simple"] * x.loc[i, "ret"]
        gross_ret.append(gr)

        c = change * 0.0005
        cost.append(c)

        sr = gr - c
        strategy_ret.append(sr)

        eq = equity[-1] * (1.0 + sr)
        equity.append(eq)

    x["position_change_mr_simple"] = position_change
    x["trade_flag_mr_simple"] = trade_flag
    x["gross_ret_mr_simple"] = gross_ret
    x["cost_mr_simple"] = cost
    x["strategy_ret_mr_simple"] = strategy_ret
    x["equity_mr_simple"] = equity

    exec_rows = []
    seen = set()

    for i in range(len(df)):
        h = df.loc[i, "hour"]
        if h in seen:
            continue
        seen.add(h)
        row = {}
        row["exec_hour"] = h
        row["exec_price"] = df.loc[i, "price_usdc_per_weth"]
        row["exec_gas_cost"] = df.loc[i, "gas_cost_eth"]
        exec_rows.append(row)

    exec_events = pd.DataFrame(exec_rows)

    mr_real = x[["hour", "signal_mr"]].copy()
    real_position = [0]
    for i in range(1, len(mr_real)):
        real_position.append(mr_real.loc[i - 1, "signal_mr"])
    mr_real["position"] = real_position

    mr_real = mr_real.merge(exec_events, left_on="hour", right_on="exec_hour", how="left")

    exec_ret = [0.0]
    for i in range(1, len(mr_real)):
        prev_price = mr_real.loc[i - 1, "exec_price"]
        cur_price = mr_real.loc[i, "exec_price"]
        if prev_price == 0 or pd.isna(prev_price) or pd.isna(cur_price):
            exec_ret.append(0.0)
        else:
            exec_ret.append((cur_price / prev_price) - 1.0)
    mr_real["exec_ret"] = exec_ret

    position_change = [0.0]
    trade_flag = [0]
    gross_ret = [0.0]
    gas_cost_return = [0.0]
    slippage_cost_return = [0.0]
    cost = [0.0]
    strategy_ret = [0.0]
    equity = [1.0]

    for i in range(1, len(mr_real)):
        change = abs(mr_real.loc[i, "position"] - mr_real.loc[i - 1, "position"])
        position_change.append(change)

        if change > 0:
            trade_flag.append(1)
        else:
            trade_flag.append(0)

        gr = mr_real.loc[i, "position"] * mr_real.loc[i, "exec_ret"]
        gross_ret.append(gr)

        if change > 0 and not pd.isna(mr_real.loc[i, "exec_gas_cost"]):
            gas_val = mr_real.loc[i, "exec_gas_cost"] / 1.0
        else:
            gas_val = 0.0
        gas_cost_return.append(gas_val)

        slip = change * (1.0 / 10000.0)
        slippage_cost_return.append(slip)

        c = gas_val + slip
        cost.append(c)

        sr = gr - c
        strategy_ret.append(sr)

        eq = equity[-1] * (1.0 + sr)
        equity.append(eq)

    mr_real["position_change"] = position_change
    mr_real["trade_flag"] = trade_flag
    mr_real["gross_ret"] = gross_ret
    mr_real["gas_cost_return"] = gas_cost_return
    mr_real["slippage_cost_return"] = slippage_cost_return
    mr_real["cost"] = cost
    mr_real["strategy_ret"] = strategy_ret
    mr_real["equity"] = equity

    return x, mr_real

In [ ]:
def run_signal_backtest(x, df, signal_col, pref):
    x = x.copy()

    pos = [0]
    for i in range(1, len(x)):
        pos.append(x.loc[i - 1, signal_col])

    x["position_" + pref + "_simple"] = pos

    pos_ch = [0.0]
    trade_flag = [0]
    gross_ret = [0.0]
    cost = [0.0]
    strategy_ret = [0.0]
    equity = [1.0]

    for i in range(1, len(x)):
        change = abs(x.loc[i, "position_" + pref + "_simple"] - x.loc[i - 1, "position_" + pref + "_simple"])
        pos_ch.append(change)

        if change > 0:
            trade_flag.append(1)
        else:
            trade_flag.append(0)

        gr = x.loc[i, "position_" + pref + "_simple"] * x.loc[i, "ret"]
        gross_ret.append(gr)

        c = change * 0.0005
        cost.append(c)

        sr = gr - c
        strategy_ret.append(sr)

        eq = equity[-1] * (1.0 + sr)
        equity.append(eq)

    x["position_change_" + pref + "_simple"] = pos_ch
    x["trade_flag_" + pref + "_simple"] = trade_flag
    x["gross_ret_" + pref + "_simple"] = gross_ret
    x["cost_" + pref + "_simple"] = cost
    x["strategy_ret_" + pref + "_simple"] = strategy_ret
    x["equity_" + pref + "_simple"] = equity

    exec_rows = []
    seen = set()

    for i in range(len(df)):
        h = df.loc[i, "hour"]
        if h in seen:
            continue
        seen.add(h)

        row = {}
        row["exec_hour"] = h
        row["exec_price"] = df.loc[i, "price_usdc_per_weth"]
        row["exec_gas_cost"] = df.loc[i, "gas_cost_eth"]
        exec_rows.append(row)

    exec_events = pd.DataFrame(exec_rows)

    real = x[["hour", signal_col]].copy()

    real_pos = [0]
    for i in range(1, len(real)):
        real_pos.append(real.loc[i - 1, signal_col])

    real["position"] = real_pos
    real = real.merge(exec_events, left_on="hour", right_on="exec_hour", how="left")

    exec_ret = [0.0]
    for i in range(1, len(real)):
        p1 = real.loc[i - 1, "exec_price"]
        p2 = real.loc[i, "exec_price"]

        if p1 == 0 or pd.isna(p1) or pd.isna(p2):
            exec_ret.append(0.0)
        else:
            exec_ret.append((p2 / p1) - 1.0)

    real["exec_ret"] = exec_ret

    pos_ch = [0.0]
    trade_flag = [0]
    gross_ret = [0.0]
    gas_cost_return = [0.0]
    slippage_cost_return = [0.0]
    cost = [0.0]
    strategy_ret = [0.0]
    equity = [1.0]

    for i in range(1, len(real)):
        change = abs(real.loc[i, "position"] - real.loc[i - 1, "position"])
        pos_ch.append(change)

        if change > 0:
            trade_flag.append(1)
        else:
            trade_flag.append(0)

        gr = real.loc[i, "position"] * real.loc[i, "exec_ret"]
        gross_ret.append(gr)

        if change > 0 and not pd.isna(real.loc[i, "exec_gas_cost"]):
            gas_val = real.loc[i, "exec_gas_cost"] / 1.0
        else:
            gas_val = 0.0

        gas_cost_return.append(gas_val)

        slip = change * (1.0 / 10000.0)
        slippage_cost_return.append(slip)

        c = gas_val + slip
        cost.append(c)

        sr = gr - c
        strategy_ret.append(sr)

        eq = equity[-1] * (1.0 + sr)
        equity.append(eq)

    real["position_change"] = pos_ch
    real["trade_flag"] = trade_flag
    real["gross_ret"] = gross_ret
    real["gas_cost_return"] = gas_cost_return
    real["slippage_cost_return"] = slippage_cost_return
    real["cost"] = cost
    real["strategy_ret"] = strategy_ret
    real["equity"] = equity

    return x, real

In [ ]:
def run_rsi_strategy(hourly, df):
    x = hourly.copy()

    rsi = []
    for i in range(len(x)):
        if i < 14:
            rsi.append(np.nan)
        else:
            gains = []
            losses = []

            for j in range(i - 13, i + 1):
                r = x.loc[j, "ret"]
                if r > 0:
                    gains.append(r)
                    losses.append(0)
                else:
                    gains.append(0)
                    losses.append(abs(r))

            avg_gain = np.mean(gains)
            avg_loss = np.mean(losses)

            if avg_loss == 0:
                rsi.append(100)
            else:
                rs = avg_gain / avg_loss
                rsi.append(100 - (100 / (1 + rs)))

    x["rsi"] = rsi

    sig = []
    for i in range(len(x)):
        z = x.loc[i, "rsi"]

        if pd.isna(z):
            sig.append(0)
        elif z < 30:
            sig.append(1)
        elif z > 70:
            sig.append(-1)
        else:
            sig.append(0)

    x["signal_rsi"] = sig

    return run_signal_backtest(x, df, "signal_rsi", "rsi")

In [ ]:
def run_bollinger_strategy(hourly, df):
    x = hourly.copy()

    sig = []

    for i in range(len(x)):
        if i < 20:
            sig.append(0)
        else:
            kusok = x.loc[i-19:i, "close"]
            m = kusok.mean()
            s = kusok.std()
            price = x.loc[i, "close"]

            upper = m + 2 * s
            lower = m - 2 * s

            if price < lower:
                sig.append(1)
            elif price > upper:
                sig.append(-1)
            else:
                sig.append(0)

    x["signal_boll"] = sig

    return run_signal_backtest(x, df, "signal_boll", "boll")

In [ ]:
def run_breakout_strategy(hourly, df):
    x = hourly.copy()

    sig = []

    for i in range(len(x)):
        if i < 24:
            sig.append(0)
        else:
            high_prev = x.loc[i-24:i-1, "high"].max()
            low_prev = x.loc[i-24:i-1, "low"].min()
            price = x.loc[i, "close"]

            if price > high_prev:
                sig.append(1)
            elif price < low_prev:
                sig.append(-1)
            else:
                sig.append(0)

    x["signal_breakout"] = sig

    return run_signal_backtest(x, df, "signal_breakout", "breakout")

In [ ]:
def run_momentum_strategy(hourly, df):
    x = hourly.copy()

    sig = []

    for i in range(len(x)):
        if i < 24:
            sig.append(0)
        else:
            old_price = x.loc[i - 24, "close"]
            cur_price = x.loc[i, "close"]

            mom = cur_price / old_price - 1.0

            if mom > 0.02:
                sig.append(1)
            elif mom < -0.02:
                sig.append(-1)
            else:
                sig.append(0)

    x["signal_mom"] = sig

    return run_signal_backtest(x, df, "signal_mom", "mom")

In [ ]:
def run_roc_strategy(hourly, df):
    x = hourly.copy()

    sig = []

    for i in range(len(x)):
        if i < 12:
            sig.append(0)
        else:
            p_old = x.loc[i - 12, "close"]
            p_new = x.loc[i, "close"]

            roc = (p_new - p_old) / p_old

            if roc > 0.01:
                sig.append(1)
            elif roc < -0.01:
                sig.append(-1)
            else:
                sig.append(0)

    x["signal_roc"] = sig

    return run_signal_backtest(x, df, "signal_roc", "roc")

In [ ]:
def run_ema_strategy(hourly, df):
    x = hourly.copy()

    ema_fast = []
    ema_slow = []

    alpha_fast = 2 / (24 + 1)
    alpha_slow = 2 / (72 + 1)

    for i in range(len(x)):
        price = x.loc[i, "close"]

        if i == 0:
            ema_fast.append(price)
            ema_slow.append(price)
        else:
            ema_fast.append(alpha_fast * price + (1 - alpha_fast) * ema_fast[-1])
            ema_slow.append(alpha_slow * price + (1 - alpha_slow) * ema_slow[-1])

    x["ema_fast"] = ema_fast
    x["ema_slow"] = ema_slow

    sig = []

    for i in range(len(x)):
        if x.loc[i, "ema_fast"] > x.loc[i, "ema_slow"]:
            sig.append(1)
        elif x.loc[i, "ema_fast"] < x.loc[i, "ema_slow"]:
            sig.append(-1)
        else:
            sig.append(0)

    x["signal_ema"] = sig

    return run_signal_backtest(x, df, "signal_ema", "ema")

In [ ]:
def run_channel_reversion_strategy(hourly, df):
    x = hourly.copy()

    sig = []

    for i in range(len(x)):
        if i < 48:
            sig.append(0)
        else:
            high_ch = x.loc[i-48:i-1, "high"].max()
            low_ch = x.loc[i-48:i-1, "low"].min()
            price = x.loc[i, "close"]

            center = (high_ch + low_ch) / 2

            if price > high_ch:
                sig.append(-1)
            elif price < low_ch:
                sig.append(1)
            elif abs(price - center) / center < 0.003:
                sig.append(0)
            else:
                sig.append(0)

    x["signal_channel_mr"] = sig

    return run_signal_backtest(x, df, "signal_channel_mr", "channel")

In [ ]:
def run_volatility_breakout_strategy(hourly, df):
    x = hourly.copy()

    sig = []

    for i in range(len(x)):
        if i < 24:
            sig.append(0)
        else:
            old = x.loc[i-24:i-1, "close"]
            srednee = old.mean()
            vol = old.std()
            price = x.loc[i, "close"]

            if price > srednee + 1.5 * vol:
                sig.append(1)
            elif price < srednee - 1.5 * vol:
                sig.append(-1)
            else:
                sig.append(0)

    x["signal_volbr"] = sig

    return run_signal_backtest(x, df, "signal_volbr", "volbr")

Считаем стратегии

In [ ]:
hourly_ma, ma_real = run_ma_strategy(hourly, df)
hourly_mr, mr_real = run_mr_strategy(hourly, df)

hourly_rsi, rsi_real = run_rsi_strategy(hourly, df)
hourly_boll, boll_real = run_bollinger_strategy(hourly, df)
hourly_breakout, breakout_real = run_breakout_strategy(hourly, df)
hourly_mom, mom_real = run_momentum_strategy(hourly, df)
hourly_roc, roc_real = run_roc_strategy(hourly, df)
hourly_volbr, volbr_real = run_volatility_breakout_strategy(hourly, df)
hourly_ema, ema_real = run_ema_strategy(hourly, df)
hourly_channel, channel_real = run_channel_reversion_strategy(hourly, df)

In [ ]:
def plot_strategy_comparison(simple_df, real_df, simple_equity_col, title, filename):
    fig, ax = plt.subplots(figsize=(6.2, 3.4))

    ax.plot(simple_df["hour"], simple_df[simple_equity_col], label="Simplified")
    ax.plot(real_df["hour"], real_df["equity"], label="Realistic")

    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Equity")
    ax.legend()

    fig.savefig("/Users/nick/CodexProjects/Course_Project/figures_new/" + filename + ".png", bbox_inches="tight")
    plt.show()

In [ ]:
plots = [
    (hourly_ma, ma_real, "equity_ma_simple", "Cumulative performance: MA crossover", "ma_crossover_comparison"),
    (hourly_mr, mr_real, "equity_mr_simple", "Cumulative performance: Mean reversion", "mean_reversion_comparison"),
    (hourly_rsi, rsi_real, "equity_rsi_simple", "Cumulative performance: RSI", "rsi_comparison"),
    (hourly_boll, boll_real, "equity_boll_simple", "Cumulative performance: Bollinger", "bollinger_comparison"),
    (hourly_breakout, breakout_real, "equity_breakout_simple", "Cumulative performance: Breakout", "breakout_comparison"),
    (hourly_mom, mom_real, "equity_mom_simple", "Cumulative performance: Momentum", "momentum_comparison"),
    (hourly_roc, roc_real, "equity_roc_simple", "Cumulative performance: ROC", "roc_comparison"),
    (hourly_volbr, volbr_real, "equity_volbr_simple", "Cumulative performance: Volatility breakout", "volatility_breakout_comparison"),
    (hourly_ema, ema_real, "equity_ema_simple", "Cumulative performance: EMA crossover", "ema_comparison"),
    (hourly_channel, channel_real, "equity_channel_simple", "Cumulative performance: Channel reversion", "channel_reversion_comparison"),
]

for simple_df, real_df, simple_equity_col, title, filename in plots:
    plot_strategy_comparison(simple_df, real_df, simple_equity_col, title, filename)

In [ ]:
def calc_sharpe(ret_col):
    r = []
    for x in ret_col:
        if not pd.isna(x):
            r.append(x)

    if len(r) == 0 or np.std(r) == 0:
        return np.nan

    return np.sqrt(24 * 365) * np.mean(r) / np.std(r)


def calc_max_drawdown(equity_col):
    eq = list(equity_col)

    if len(eq) == 0:
        return np.nan

    running_max = eq[0]
    max_dd = 0.0

    for x in eq:
        if x > running_max:
            running_max = x

        dd = x / running_max - 1.0

        if dd < max_dd:
            max_dd = dd

    return max_dd


def calc_total_return(equity_col):
    eq = list(equity_col)

    if len(eq) == 0:
        return np.nan

    return eq[-1] - 1.0


def get_strategy_result(strategy_name, model_name, df_res, ret_col, equity_col, trades_col):
    total_return = calc_total_return(df_res[equity_col])
    sharpe = calc_sharpe(df_res[ret_col])
    max_dd = calc_max_drawdown(df_res[equity_col])
    n_trades = int(df_res[trades_col].sum())

    return {
        "Strategy": strategy_name,
        "Execution Model": model_name,
        "Total Return": total_return,
        "Sharpe": sharpe,
        "Max Drawdown": max_dd,
        "Number of Trades": n_trades
    }

In [ ]:
all_results = []

all_results.append(
    get_strategy_result(
        "MA crossover",
        "Simplified",
        hourly_ma,
        "strategy_ret_ma_simple",
        "equity_ma_simple",
        "trade_flag_ma_simple"
    )
)

all_results.append(
    get_strategy_result(
        "MA crossover",
        "Realistic",
        ma_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "Mean reversion",
        "Simplified",
        hourly_mr,
        "strategy_ret_mr_simple",
        "equity_mr_simple",
        "trade_flag_mr_simple"
    )
)

all_results.append(
    get_strategy_result(
        "Mean reversion",
        "Realistic",
        mr_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "RSI",
        "Simplified",
        hourly_rsi,
        "strategy_ret_rsi_simple",
        "equity_rsi_simple",
        "trade_flag_rsi_simple"
    )
)

all_results.append(
    get_strategy_result(
        "RSI",
        "Realistic",
        rsi_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "Bollinger",
        "Simplified",
        hourly_boll,
        "strategy_ret_boll_simple",
        "equity_boll_simple",
        "trade_flag_boll_simple"
    )
)

all_results.append(
    get_strategy_result(
        "Bollinger",
        "Realistic",
        boll_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "Breakout",
        "Simplified",
        hourly_breakout,
        "strategy_ret_breakout_simple",
        "equity_breakout_simple",
        "trade_flag_breakout_simple"
    )
)

all_results.append(
    get_strategy_result(
        "Breakout",
        "Realistic",
        breakout_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "Momentum",
        "Simplified",
        hourly_mom,
        "strategy_ret_mom_simple",
        "equity_mom_simple",
        "trade_flag_mom_simple"
    )
)

all_results.append(
    get_strategy_result(
        "Momentum",
        "Realistic",
        mom_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "ROC",
        "Simplified",
        hourly_roc,
        "strategy_ret_roc_simple",
        "equity_roc_simple",
        "trade_flag_roc_simple"
    )
)

all_results.append(
    get_strategy_result(
        "ROC",
        "Realistic",
        roc_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "Volatility breakout",
        "Simplified",
        hourly_volbr,
        "strategy_ret_volbr_simple",
        "equity_volbr_simple",
        "trade_flag_volbr_simple"
    )
)

all_results.append(
    get_strategy_result(
        "Volatility breakout",
        "Realistic",
        volbr_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "EMA crossover",
        "Simplified",
        hourly_ema,
        "strategy_ret_ema_simple",
        "equity_ema_simple",
        "trade_flag_ema_simple"
    )
)

all_results.append(
    get_strategy_result(
        "EMA crossover",
        "Realistic",
        ema_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

all_results.append(
    get_strategy_result(
        "Channel reversion",
        "Simplified",
        hourly_channel,
        "strategy_ret_channel_simple",
        "equity_channel_simple",
        "trade_flag_channel_simple"
    )
)

all_results.append(
    get_strategy_result(
        "Channel reversion",
        "Realistic",
        channel_real,
        "strategy_ret",
        "equity",
        "trade_flag"
    )
)

results_table = pd.DataFrame(all_results)

results_table = results_table.round({
    "Total Return": 4,
    "Sharpe": 3,
    "Max Drawdown": 4
})

results_table

In [ ]:
results_table.to_csv("/Users/nick/CodexProjects/Course_Project/figures_new/" + "strategy_results_table.csv", index=False)

In [ ]:
labels = []
sharpe_values = []
colors = []

for i in range(len(results_table)):
    name = results_table.loc[i, "Strategy"]
    model = results_table.loc[i, "Execution Model"]

    labels.append(name)
    sharpe_values.append(results_table.loc[i, "Sharpe"])

    if model == "Realistic":
        colors.append("#F28E2B")
    else:
        colors.append("tab:blue")

In [ ]:
part1_idx = list(range(0, 10))

fig, ax = plt.subplots(figsize=(9.0, 4.2))

ax.bar(
    [labels[i] + "\n" + results_table.loc[i, "Execution Model"] for i in part1_idx],
    [sharpe_values[i] for i in part1_idx],
    color=[colors[i] for i in part1_idx],
    edgecolor="black"
)

ax.set_title("Sharpe ratios across strategies and execution models (part 1)")
ax.set_ylabel("Sharpe ratio")
ax.tick_params(axis="x", rotation=25)

fig.tight_layout()
fig.savefig("/Users/nick/CodexProjects/Course_Project/figures_new/strategy_sharpe_comparison_part1.png", bbox_inches="tight")
plt.show()

In [ ]:
part2_idx = list(range(10, len(results_table)))

fig, ax = plt.subplots(figsize=(9.0, 4.2))

ax.bar(
    [labels[i] + "\n" + results_table.loc[i, "Execution Model"] for i in part2_idx],
    [sharpe_values[i] for i in part2_idx],
    color=[colors[i] for i in part2_idx],
    edgecolor="black"
)

ax.set_title("Sharpe ratios across strategies and execution models (part 2)")
ax.set_ylabel("Sharpe ratio")
ax.tick_params(axis="x", rotation=25)

fig.tight_layout()
fig.savefig("/Users/nick/CodexProjects/Course_Project/figures_new/strategy_sharpe_comparison_part2.png", bbox_inches="tight")
plt.show()